# 03 · Error Analysis — RoBERTa-base en SQuAD v2

Usa las predicciones de validation guardadas por `02_train_roberta.ipynb`
(`predictions_val.json` en Drive), así que **no necesita GPU ni volver a correr el modelo**.

Etapas:
1. **Búsqueda de threshold** de "sin respuesta" → `threshold_search.png`.
2. **Clasificación de errores** con la normalización oficial de SQuAD.
3. Ejemplos de predicciones **correctas e incorrectas**, para contestables y no contestables.
4. Errores por tipo de pregunta y por longitud del contexto.
5. Guardado de `error_analysis.json`.
6. Generación automática de `README.md` y `ROBERTA_COMPARISON.md` a partir de los JSON reales.

## 1. Instalación, rutas y datos

In [ ]:
# ⚠️ Es normal que pip muestre un aviso de 'dependency conflicts' con gradio/diffusers:
#    este notebook no usa esos paquetes. Tras esta celda, la primera vez:
#    Entorno de ejecución → Reiniciar sesión, y luego Ejecutar todas de nuevo.
%pip install -q "datasets>=3.0" "evaluate>=0.4"

In [ ]:
import os, sys

# ── Rutas ─────────────────────────────────────────────────────────────
# En Colab todo se guarda en Google Drive para que no se pierda si la sesión
# se desconecta. Fuera de Colab se usa la raíz del repositorio.
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = "/content/drive/MyDrive/roberta_squad2"
else:
    BASE_DIR = os.path.abspath("..")   # el notebook vive en pipeline_roberta/

LOGS_DIR   = os.path.join(BASE_DIR, "logs_roberta")                 # resultados para GitHub
MODEL_DIR  = os.path.join(BASE_DIR, "roberta-base-squad2-final")    # modelo final (NO va a GitHub)
CKPT_DIR   = os.path.join(BASE_DIR, "checkpoints")                  # checkpoints intermedios
PRED_FILE  = os.path.join(BASE_DIR, "predictions_val.json")         # predicciones de validación
os.makedirs(LOGS_DIR, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("LOGS_DIR:", LOGS_DIR)

In [ ]:
import json, re, string, collections, random
import numpy as np
import matplotlib.pyplot as plt
from datasets import load_dataset
import evaluate

assert os.path.exists(PRED_FILE), f"No existe {PRED_FILE}. Ejecuta primero 02_train_roberta.ipynb."
results = json.load(open(PRED_FILE, encoding="utf-8"))
eval_results = json.load(open(os.path.join(LOGS_DIR, "eval_results.json"), encoding="utf-8"))
REPORTED_THRESHOLD = eval_results["null_threshold"]

raw_val = load_dataset("rajpurkar/squad_v2", split="validation")
by_id = {ex["id"]: ex for ex in raw_val}
assert len(results) == len(raw_val)
print(f"{len(results):,} predicciones cargadas | threshold usado en eval_results.json: {REPORTED_THRESHOLD}")

## 2. Búsqueda de threshold

`score_diff = score(mejor span) − score(<s>)`. Se responde con el span si
`score_diff > threshold`; si no, se devuelve `""`.

- Threshold **bajo** → responde casi siempre: sube F1 en contestables, baja en no contestables.
- Threshold **alto** → se abstiene más: lo contrario.

⚠️ El threshold se elige sobre el mismo validation set que se reporta, así que el
"mejor F1" de esta búsqueda es **optimista**. Las métricas oficiales del modelo son las
de `eval_results.json` (threshold fijo = 0.0). El valor encontrado aquí sirve para
configurar `NULL_THRESHOLD` en la API.

In [ ]:
squad_metric = evaluate.load("squad_v2")
refs = [{"id": ex["id"], "answers": ex["answers"]} for ex in raw_val]

def apply_threshold(threshold):
    return {r["id"]: (r["best_span_text"] if r["score_diff"] > threshold else "") for r in results}

def scores_at(threshold):
    preds = [{"id": k, "prediction_text": v, "no_answer_probability": 0.0}
             for k, v in apply_threshold(threshold).items()]
    return squad_metric.compute(predictions=preds, references=refs)

thresholds = np.round(np.arange(-5.0, 5.01, 0.5), 2).tolist()
search = []
for t in thresholds:
    s = scores_at(t)
    search.append({"threshold": t, "exact": s["exact"], "f1": s["f1"],
                   "HasAns_f1": s["HasAns_f1"], "NoAns_f1": s["NoAns_f1"]})
    print(f"t={t:+.1f}  EM={s['exact']:.2f}  F1={s['f1']:.2f}  HasAns F1={s['HasAns_f1']:.2f}  NoAns F1={s['NoAns_f1']:.2f}")

best = max(search, key=lambda r: r["f1"])
print(f"\nMejor threshold por F1 global: {best['threshold']:+.1f} → F1 {best['f1']:.2f}")

In [ ]:
plt.figure(figsize=(11, 5))
T = [r["threshold"] for r in search]
plt.plot(T, [r["f1"] for r in search], "o-", label="F1 global", color="#1565C0", lw=2)
plt.plot(T, [r["exact"] for r in search], "s-", label="EM global", color="#6A1B9A")
plt.plot(T, [r["HasAns_f1"] for r in search], "--", label="F1 contestables", color="#2E7D32")
plt.plot(T, [r["NoAns_f1"] for r in search], "--", label="F1 no contestables", color="#C62828")
plt.axvline(best["threshold"], color="black", ls=":", label=f"Mejor threshold = {best['threshold']:+.1f}")
plt.axvline(REPORTED_THRESHOLD, color="gray", ls="-", alpha=0.4, label=f"Threshold reportado = {REPORTED_THRESHOLD}")
plt.xlabel("Threshold (score_span − score_nulo)"); plt.ylabel("Score (%)")
plt.title("RoBERTa-base / SQuAD v2 validation — métricas vs. threshold de no-respuesta")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(os.path.join(LOGS_DIR, "threshold_search.png"), dpi=150)
plt.show()

## 3. Clasificación de errores

Se usa la **normalización oficial de SQuAD** (minúsculas, sin puntuación, sin artículos
*a/an/the*, espacios colapsados), así que `"the Normans"` y `"Normans"` cuentan igual.

Categorías (con el threshold reportado, para que coincidan con `eval_results.json`):

| Tipo de pregunta | Categoría | Significado |
|---|---|---|
| contestable | `correct` | EM = 1 |
| contestable | `partial_overlap` | span con solapamiento parcial (0 < F1 < 1) |
| contestable | `wrong_span` | span sin ninguna palabra en común (F1 = 0) |
| contestable | `false_negative` | el modelo dijo "sin respuesta" y sí había |
| no contestable | `correct` | el modelo devolvió `""` |
| no contestable | `false_positive` | el modelo inventó un span |

In [ ]:
ANALYSIS_THRESHOLD = REPORTED_THRESHOLD

def normalize_answer(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def token_f1(pred, gold):
    p, g = normalize_answer(pred).split(), normalize_answer(gold).split()
    if not p or not g:
        return float(p == g)
    same = sum((collections.Counter(p) & collections.Counter(g)).values())
    if same == 0:
        return 0.0
    prec, rec = same / len(p), same / len(g)
    return 2 * prec * rec / (prec + rec)

QTYPES = ["what", "who", "how", "when", "which", "where", "why"]
def qtype(q):
    w = q.strip().lower().split()
    return w[0] if w and w[0] in QTYPES else "other"

preds = apply_threshold(ANALYSIS_THRESHOLD)
rows = []
for r in results:
    ex = by_id[r["id"]]
    pred = preds[r["id"]]
    golds = ex["answers"]["text"]
    answerable = len(golds) > 0
    gold_list = golds if answerable else [""]
    em = max(float(normalize_answer(pred) == normalize_answer(g)) for g in gold_list)
    f1 = max(token_f1(pred, g) for g in gold_list)
    if em == 1:
        cat = "correct"
    elif not answerable:
        cat = "false_positive"
    elif pred == "":
        cat = "false_negative"
    elif f1 > 0:
        cat = "partial_overlap"
    else:
        cat = "wrong_span"
    rows.append({"id": r["id"], "question": ex["question"], "context": ex["context"],
                 "gold": golds, "prediction": pred, "answerable": answerable,
                 "category": cat, "em": em, "f1": round(f1, 3),
                 "score_diff": round(r["score_diff"], 3), "qtype": qtype(ex["question"])})

# Verificación: los promedios deben coincidir con la métrica oficial de eval_results.json
em_all, f1_all = 100 * np.mean([x["em"] for x in rows]), 100 * np.mean([x["f1"] for x in rows])
print(f"EM {em_all:.2f} / F1 {f1_all:.2f}  (eval_results.json: "
      f"{eval_results['overall']['exact_match']:.2f} / {eval_results['overall']['f1']:.2f})")

In [ ]:
def breakdown(subset):
    c = collections.Counter(x["category"] for x in subset)
    return {k: c[k] for k in sorted(c)}

answerable_rows   = [x for x in rows if x["answerable"]]
unanswerable_rows = [x for x in rows if not x["answerable"]]
errors = [x for x in rows if x["category"] != "correct"]

print(f"Errores totales: {len(errors):,} de {len(rows):,} ({len(errors)/len(rows):.1%})\n")
for name, subset in [("Contestables", answerable_rows), ("No contestables", unanswerable_rows)]:
    print(f"{name} (n={len(subset):,})")
    for k, v in breakdown(subset).items():
        print(f"   {k:<16} {v:>6,}  ({v/len(subset):.1%})")

## 4. Errores por tipo de pregunta y por longitud de contexto

In [ ]:
def error_rate_by(key_fn, subset):
    groups = collections.defaultdict(list)
    for x in subset:
        groups[key_fn(x)].append(x["category"] != "correct")
    return {k: {"n": len(v), "error_rate_pct": round(100 * np.mean(v), 2)} for k, v in groups.items()}

by_qtype = error_rate_by(lambda x: x["qtype"], rows)
by_qtype = dict(sorted(by_qtype.items(), key=lambda kv: -kv[1]["n"]))

ctx_len = np.array([len(x["context"]) for x in rows])
edges = np.percentile(ctx_len, [25, 50, 75]).astype(int)
def ctx_bin(x):
    n = len(x["context"])
    if n <= edges[0]: return f"Q1 (≤{edges[0]} chars)"
    if n <= edges[1]: return f"Q2 (≤{edges[1]})"
    if n <= edges[2]: return f"Q3 (≤{edges[2]})"
    return f"Q4 (>{edges[2]})"
by_ctx = dict(sorted(error_rate_by(ctx_bin, rows).items()))

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].bar(by_qtype.keys(), [v["error_rate_pct"] for v in by_qtype.values()], color="#1565C0")
axes[0].set_title("Tasa de error por tipo de pregunta"); axes[0].set_ylabel("% errores")
axes[1].bar(by_ctx.keys(), [v["error_rate_pct"] for v in by_ctx.values()], color="#EF6C00")
axes[1].set_title("Tasa de error por cuartil de longitud de contexto"); axes[1].set_ylabel("% errores")
plt.tight_layout(); plt.show()   # solo se muestra en el notebook; los datos van a error_analysis.json
print(json.dumps({"by_question_type": by_qtype, "by_context_length": by_ctx}, indent=2, ensure_ascii=False))

## 5. Ejemplos de predicciones correctas e incorrectas

In [ ]:
random.seed(42)
def sample(cat, answerable, k=8):
    pool = [x for x in rows if x["category"] == cat and x["answerable"] == answerable]
    picked = random.sample(pool, min(k, len(pool)))
    return [{"id": x["id"], "question": x["question"], "context": x["context"][:400] + "...",
             "gold": x["gold"], "prediction": x["prediction"], "f1": x["f1"],
             "score_diff": x["score_diff"]} for x in picked]

examples = {
    "correct_answerable":   sample("correct", True),
    "correct_unanswerable": sample("correct", False),
    "partial_overlap":      sample("partial_overlap", True),
    "wrong_span":           sample("wrong_span", True),
    "false_negative":       sample("false_negative", True),
    "false_positive":       sample("false_positive", False),
}

for name, items in examples.items():
    print(f"\n=== {name} ({len(items)} ejemplos) ===")
    for x in items[:3]:
        print(f"Q: {x['question']}")
        print(f"   gold: {x['gold'] or '(sin respuesta)'} | pred: {x['prediction']!r} | score_diff={x['score_diff']}")

## 6. Guardar `error_analysis.json`

In [ ]:
error_analysis = {
    "analysis_threshold": ANALYSIS_THRESHOLD,
    "normalization": "oficial SQuAD (lower, sin puntuación, sin artículos, espacios colapsados)",
    "total_examples": len(rows),
    "total_errors": len(errors),
    "error_rate_pct": round(100 * len(errors) / len(rows), 2),
    "breakdown_answerable": breakdown(answerable_rows),
    "breakdown_unanswerable": breakdown(unanswerable_rows),
    "by_question_type": by_qtype,
    "by_context_length": by_ctx,
    "threshold_search": {
        "best_threshold": best["threshold"],
        "best_f1": best["f1"],
        "best_exact": best["exact"],
        "note": "Elegido sobre el mismo validation set: valor optimista; úsalo para NULL_THRESHOLD en la API.",
        "grid": search,
    },
    "examples": examples,
}
with open(os.path.join(LOGS_DIR, "error_analysis.json"), "w", encoding="utf-8") as f:
    json.dump(error_analysis, f, indent=2, ensure_ascii=False)
print("✅ Guardado:", os.path.join(LOGS_DIR, "error_analysis.json"))

## 7. Generar `README.md` y `ROBERTA_COMPARISON.md`

Se escriben **solo con los valores de los JSON** de `logs_roberta/`. Los números de
ModernBERT de la comparación vienen de `logs_modernBert/` en el repositorio del equipo.

In [ ]:
tr = json.load(open(os.path.join(LOGS_DIR, "train_results.json"), encoding="utf-8"))
ev = eval_results
eda = json.load(open(os.path.join(LOGS_DIR, "eda_stats.json"), encoding="utf-8"))
cfg, lat = tr["config"], ev["cpu_latency"]
f2 = lambda v: f"{v:.2f}"

readme = f"""# logs_roberta — RoBERTa-base fine-tuned en SQuAD v2

Resultados reales del fine-tuning de `{tr['model_checkpoint']}` para Extractive QA en SQuAD v2.
Generado automáticamente por `pipeline_roberta/03_error_analysis_roberta.ipynb`.

## Resultados (validation, {ev['num_examples']:,} preguntas, threshold = {ev['null_threshold']})

| Subconjunto | EM | F1 | n |
|---|---|---|---|
| **Global** | **{f2(ev['overall']['exact_match'])}** | **{f2(ev['overall']['f1'])}** | {ev['overall']['count']:,} |
| Contestables | {f2(ev['answerable']['exact_match'])} | {f2(ev['answerable']['f1'])} | {ev['answerable']['count']:,} |
| No contestables | {f2(ev['unanswerable']['exact_match'])} | {f2(ev['unanswerable']['f1'])} | {ev['unanswerable']['count']:,} |

Mejor threshold en la búsqueda: **{error_analysis['threshold_search']['best_threshold']:+.1f}**
(F1 {f2(error_analysis['threshold_search']['best_f1'])}; optimista porque se eligió sobre validation).

## Entrenamiento

| Parámetro | Valor |
|---|---|
| Modelo base | `{tr['model_checkpoint']}` |
| Train | {tr['data']['train_examples']:,} ejemplos ({tr['data']['train_features']:,} ventanas) |
| Validation | {tr['data']['val_examples']:,} ejemplos |
| Epochs | {cfg['num_train_epochs']} |
| Learning rate | {cfg['learning_rate']} ({cfg['lr_scheduler_type']}, warmup {cfg['warmup_ratio']:.0%}) |
| Batch | {cfg['per_device_train_batch_size']} × {cfg['gradient_accumulation_steps']} = {cfg['effective_batch_size']} efectivo |
| max_seq_length / doc_stride | {cfg['max_seq_length']} / {cfg['doc_stride']} |
| GPU | {tr['hardware']['gpu']} ({tr['hardware']['precision']}) |
| Tiempo de entrenamiento | {tr['training_time']['total_hours']} h ({tr['training_time']['num_sessions']} sesión(es)) |
| Parámetros | {tr['num_parameters']:,} |
| Tamaño de pesos | {tr['model_weights_mb']} MB |
| Latencia CPU (media / p95) | {lat['mean_ms']} / {lat['p95_ms']} ms ({lat['environment']}, {lat['cpu_count']} CPUs) |

## Archivos

| Archivo | Contenido |
|---|---|
| `eda_squad2.png`, `eda_stats.json` | Análisis exploratorio de SQuAD v2 |
| `loss.png` | Pérdida de entrenamiento y validación |
| `train_results.json` | Configuración exacta, hardware, tiempo, tamaño |
| `eval_results.json` | EM/F1 global, contestables y no contestables; latencia CPU |
| `threshold_search.png` | Métricas vs. threshold de no-respuesta |
| `error_analysis.json` | Tipos de error y ejemplos correctos/incorrectos |
| `ROBERTA_COMPARISON.md` | Ficha comparativa con ModernBERT |

![EDA](eda_squad2.png)
![Loss](loss.png)
![Threshold](threshold_search.png)
"""

MB = {"name": "ModernBERT-base (Celeste)", "train": "30,000", "epochs": 2, "em": 77.67, "f1": 81.04,
      "has_em": 72.39, "has_f1": 79.14, "no_em": 82.94, "no_f1": 82.94,
      "params": "149,606,402", "size": "572 MB", "time": "2.85 h", "gpu": "RTX 3050 Ti 4GB"}

comparison = f"""# Ficha comparativa — RoBERTa-base vs. ModernBERT-base

Valores de RoBERTa-base: `logs_roberta/*.json`. Valores de ModernBERT-base: `logs_modernBert/`
(documentados por Celeste). Ambos evaluados en el validation completo de SQuAD v2 con `evaluate.load("squad_v2")`.

| | RoBERTa-base | {MB['name']} |
|---|---|---|
| Checkpoint | `{tr['model_checkpoint']}` | `answerdotai/ModernBERT-base` |
| Ejemplos de train | {tr['data']['train_examples']:,} | {MB['train']} |
| Epochs | {cfg['num_train_epochs']} | {MB['epochs']} |
| LR / warmup | {cfg['learning_rate']} / {cfg['warmup_ratio']:.0%} | 3e-05 / 10% |
| Batch efectivo | {cfg['effective_batch_size']} | 32 |
| max_seq_length / stride | {cfg['max_seq_length']} / {cfg['doc_stride']} | 384 / 128 |
| GPU | {tr['hardware']['gpu']} | {MB['gpu']} |
| Tiempo de entrenamiento | {tr['training_time']['total_hours']} h | {MB['time']} |
| **EM global** | **{f2(ev['overall']['exact_match'])}** | **{MB['em']}** |
| **F1 global** | **{f2(ev['overall']['f1'])}** | **{MB['f1']}** |
| EM / F1 contestables | {f2(ev['answerable']['exact_match'])} / {f2(ev['answerable']['f1'])} | {MB['has_em']} / {MB['has_f1']} |
| EM / F1 no contestables | {f2(ev['unanswerable']['exact_match'])} / {f2(ev['unanswerable']['f1'])} | {MB['no_em']} / {MB['no_f1']} |
| Parámetros | {tr['num_parameters']:,} | {MB['params']} |
| Tamaño de pesos | {tr['model_weights_mb']} MB | {MB['size']} |
| Latencia CPU media | {lat['mean_ms']} ms ({lat['environment']}) | ~1,215–1,450 ms (laptop, vía API) |

**Cuidado al comparar:** los modelos no se entrenaron con la misma cantidad de datos ni epochs,
y la latencia se midió en máquinas y formas distintas (pipeline en {lat['environment']} vs. llamadas a la API en laptop).

## Errores de RoBERTa-base (threshold {error_analysis['analysis_threshold']})

- Contestables: {error_analysis['breakdown_answerable']}
- No contestables: {error_analysis['breakdown_unanswerable']}
"""

for name, text in [("README.md", readme), ("ROBERTA_COMPARISON.md", comparison)]:
    with open(os.path.join(LOGS_DIR, name), "w", encoding="utf-8") as f:
        f.write(text)
    print("✅ Guardado:", os.path.join(LOGS_DIR, name))

## 8. Descargar `logs_roberta/`

En Colab, la carpeta ya está en Drive (`MyDrive/roberta_squad2/logs_roberta/`).
Esta celda además crea un `.zip` y lo descarga a tu computadora.

In [ ]:
import shutil
print(sorted(os.listdir(LOGS_DIR)))
zip_path = shutil.make_archive(os.path.join(BASE_DIR, "logs_roberta"), "zip", BASE_DIR, "logs_roberta")
print("ZIP:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)